# Physical Climate Risk: Tropical Cyclone Trends Affecting Western Australia (1985–2024)

**A reproducible analysis framed for AASB S2 physical-risk assessment.**

**Research question.** Has the intensity of tropical cyclones affecting Western
Australia changed between 1985 and 2024, and does it track rising sea-surface
temperatures?

**Headline finding.** Over this period the seas off WA warmed by about 0.5 °C, a
robust trend, yet WA-affecting cyclones did not get stronger. Their average
intensity drifted slightly lower and showed no positive correlation with sea-surface
temperature. The lesson for climate-risk disclosure is that you cannot extrapolate
future cyclone hazard from the recent local record; you need forward-looking
projections.

Data: IBTrACS v04r01 (NOAA NCEI), NOAA ERSSTv5, and the Bureau of Meteorology
Southern-Hemisphere database. Statistics are implemented from first principles in
`stats_utils.py` and validated in `test_stats.py`.

In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import Image, display
import analysis  # the four analysis steps, as functions

have_raw = Path('data/raw/ibtracs.SI.list.v04r01.csv').exists()
print('Raw source data present:', have_raw,
      '\n(If False, the notebook runs from the committed cleaned CSVs.)')

## 1. Data and cleaning

We load IBTrACS, keep seasons 1985–2024, and reduce the track points to one row
per storm: peak BOM 10-minute wind, minimum central pressure, peak US 1-minute
wind, and the closest approach to the WA coast. A storm is flagged
*WA-affecting* if its track passes within 500 km of the coast between the north
Kimberley and the Mid West.

**Why BOM 10-minute wind.** Agencies average wind over different periods (US =
1 minute, BOM = 10 minutes), so their winds are not interchangeable; the BOM
value runs about 12% lower. We lead with the BOM figure because the subject is WA,
and cross-check against US winds and central pressure.

**Validation.** The BOM winds match the Bureau's own database to the knot for the
major storms, the 12% US/BOM gap is exactly as the averaging difference predicts,
and 97% of the flagged WA storms also appear in the Bureau's Australian database.
Within the WA subset, BOM wind coverage is 92% (85% in the 1980s rising to 100%
recently).

In [ ]:
storms = analysis.load_storms()
wa = storms[storms.wa_affecting].copy()
print(f'{len(storms)} South Indian Ocean storms, {len(wa)} WA-affecting, '
      f'seasons {storms.season.min()}-{storms.season.max()}')
display(wa[['season','name','peak_bom_wind_kt','min_bom_pres_hpa',
            'peak_usa_wind_kt','peak_usa_sshs','min_dist_wa_km']].head(10))

## 2. Frequency and intensity by decade

About five cyclones come within 500 km of WA in an average season. Average peak
intensity has drifted **down** across the four decades, not up: mean BOM wind
fell from about 76 kt to 63 kt, and mean pressure rose (weakened).

In [ ]:
summary, all_count, wa_count = analysis.explore(storms)
display(summary.round(1))
display(Image('charts/01_annual_count.png'))
display(Image('charts/02_intensity_by_decade.png'))

## 3. Trend tests

Each annual series is tested three ways: ordinary least squares for the slope,
Mann-Kendall for non-parametric significance (the climate-standard test), and
Sen's slope for a robust magnitude.

The direction is consistently toward weaker storms. For the WA-only subset the
trend is not statistically robust (small sample, about five storms a year). Across
the whole South Indian Ocean, where the sample is large, the decline in mean wind
is significant. Two independent metrics, wind and pressure, point the same way.

In [ ]:
trend_table = analysis.trends(storms)
display(trend_table.round(3))
display(Image('charts/03_trend_wind_speed.png'))
display(Image('charts/04_trend_pressure.png'))

## 4. Rapid intensification

The share of storms that gained at least 30 kt in 24 hours (measured on US
1-minute winds, the convention for this metric) rose from about 21% in 1985–94 to
around 40% since.

**Caveat, stated openly.** Older best-track data is smoother and less frequently
sampled, which mechanically makes rapid intensification harder to detect in the
early years. Part of the apparent rise is therefore likely an observing artefact
rather than a purely physical change, so treat it as suggestive, not proven.

In [ ]:
ri_table = analysis.rapid_intensification(storms)
display(ri_table.round(1))
display(Image('charts/05_rapid_intensification.png'))

## 5. Sea-surface temperature

From ERSSTv5 we take the WA cyclone development region (10–30°S, 70–130°E),
compute monthly anomalies against the 1991–2020 climatology, and average over the
Nov–Apr cyclone season. We then correlate the seasonal anomaly with the annual
mean intensity of WA-affecting storms.

The ocean warmed significantly (about 0.16 °C per decade), yet warmer seasons were
**not** associated with stronger storms. This decoupling is the analytical heart of
the project: SST sets the available energy, but wind shear and the large-scale
circulation (ENSO, the Indian Ocean Dipole) decide whether it is realised.

In [ ]:
sst = analysis.sst_correlation(storms)
print(f"SST trend: {sst['sst_trend_decade']:+.3f} degC/decade (p={sst['sst_p']:.1e})")
print(f"SST vs WA wind:     r={sst['r_wind']:+.2f}, p={sst['p_wind']:.3f}")
print(f"SST vs WA pressure: r={sst['r_pres']:+.2f}, p={sst['p_pres']:.3f}")
display(Image('charts/06_sst_correlation.png'))

## Findings and AASB S2 implications

Frequency is flat to slightly declining. Intensity has not risen, and if anything
has weakened (significant across the basin, marginal for WA alone). Rapid
intensification appears more common, with a data caveat. And although the ocean
warmed clearly, that warming did not translate into stronger WA cyclones in the
observed record.

For AASB S2 physical-risk disclosure, beginning for large WA entities from
1 July 2026, the message is that the historical record does not support a simple
'warmer oceans, stronger storms' claim for WA, but the absence of an observed trend
is not evidence of safety. The energy ceiling has risen and projections still warn
of fewer but potentially more intense systems. Sound disclosure therefore rests on
forward-looking scenarios rather than extrapolation of the past.

See `README.md` for the full write-up and `INTERVIEW_BRIEF.md` for talking points.